# Fast MRI from Undersampled k-Space
### Reconstruction, coil redundancy and motion correction on raw knee data

An MRI scanner does not record pixels. It records the **Fourier transform of the image (k-space)**, one line at a time, and the scan time grows with the number of lines. Measuring fewer lines makes the scan faster but damages the image. This notebook uses real, fully sampled raw knee data to answer four questions:

1. **Where does the information in k-space live?**
2. **How much of k-space can be skipped, and what does skipping do to the image?**
3. **How much can classical reconstruction recover, and what makes recovery possible** (wavelet sparsity, coil redundancy, or the sampling pattern)?
4. **Can patient motion be estimated and removed using only the k-space data?**

| Phase | Content |
|---|---|
| 0 | Setup, settings and core functions |
| 1 | Data: download, verify, assemble, select slices |
| 2 | k-space anatomy |
| 3 | Where the information lives |
| 4 | Undersampling and aliasing |
| 5 | Reconstruction methods |
| 6 | Tuning on a separate subject |
| 7 | Held-out evaluation |
| 8 | Coil-count experiment |
| 9 | Motion: simulation and blind correction |
| 10 | Summary |

**Data split** (Stanford Fully Sampled 3D FSE Knees: 20 listed scans, of which 19 are currently available on mridata.org). No test subject influences any choice:

| Role | Subjects | Used for |
|---|---|---|
| Development | 0 | exploration, choosing FISTA settings (and validation for the U-Net in Notebook 2) |
| Test | 4, 9, 14, 19 | final evaluation of every method only |
| Training | the other 14 available scans | training data for the U-Net in Notebook 2 |

This is **Notebook 1 (CPU)**: the classical part of the project. It also prepares the U-Net training data, which **Notebook 2 (GPU)** reads from this notebook's saved output to train a U-Net and compare it with the methods here on exactly the same test slices and masks.

The reconstruction methods (compressed sensing with FISTA, SENSE-type coil modelling, ESPIRiT coil maps, entropy autofocus) are established techniques; references are at the end. The coil-count analysis and the blind motion estimation pipeline are the parts designed and tested here.

**How to run:** Kaggle notebook, accelerator *None* (CPU), **Internet on**. Raw files (about 1.7 GB each) are downloaded to Kaggle's temporary disk, reduced to a few slices and deleted. Every experiment stores its results in `results/`; re-running a cell reloads them instead of recomputing (set `FORCE_RERUN = True` to recompute, which is required after changing any setting). A full run is expected to take roughly 2–3 hours on a Kaggle CPU, about one hour of which is downloading and preparing the 20 subjects (Phase 7 prints its own estimate before starting). Use **Save Version → Save & Run All**, so the output can be attached to Notebook 2. Downloaded data is kept in `data/`; set `FORCE_DOWNLOAD = True` only to fetch it again.

## Phase 0 · Setup
Libraries, folders and every experiment setting in one place. Later cells only read these values.

In [ ]:
import os, sys, json, time, math, datetime, gc, shutil, subprocess
import numpy as np
import pandas as pd
import scipy
import scipy.fft as sfft
from scipy import ndimage, optimize
from functools import lru_cache
import pywt
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity
from skimage.filters import threshold_otsu
from tqdm.auto import tqdm
from IPython.display import display

ON_KAGGLE = os.path.isdir("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else os.path.abspath("work")
TEMP = "/kaggle/temp" if ON_KAGGLE else os.path.abspath("temp")
DATA_DIR, UNET_DIR, RES_DIR, FIG_DIR, CODE_DIR = (os.path.join(WORK, d) for d in ("data", "unet_data", "results", "figures", "code"))
for d in (TEMP, DATA_DIR, UNET_DIR, RES_DIR, FIG_DIR, CODE_DIR):
    os.makedirs(d, exist_ok=True)

pd.set_option("display.precision", 4)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
                     "font.size": 10, "axes.titlesize": 10, "image.cmap": "gray"})
print(f"numpy {np.__version__} | scipy {scipy.__version__} | pywt {pywt.__version__}")
print("working folder:", WORK)

**Settings**

| Setting | Meaning |
|---|---|
| `FORCE_RERUN` | recompute experiments instead of loading saved results (needed after changing any setting) |
| `FORCE_DOWNLOAD` | download and re-select the data even if it is already saved |
| `INPUT_ROOT` | where attached inputs appear; saved subject files from an earlier version of this notebook found there are reused |
| `UUIDS` | mridata.org identifiers of the 20 scans of the *Stanford Fullysampled 3D FSE Knees* project, in the order listed in the `UUIDS` table of the third-party `mridatapy` package (`mridatapy/data/mridata.py`, which states they were copied from the project page). Checked against http://mridata.org/list?project=Stanford%20Fullysampled%203D%20FSE%20Knees on 8 Nov 2026: the site lists 19 scans, all present here except index 18 (`b7d435a1-…`, download returns 404). The index in this list is the subject number used throughout. |
| `UNAVAILABLE` | subject numbers whose scan is no longer on mridata.org; never downloaded |
| `TUNE_SUBJECT` | development subject: exploration and choosing algorithm parameters |
| `TEST_SUBJECTS` | subjects used only for evaluation |
| `TRAIN_SUBJECTS` | U-Net training subjects (all remaining ones) |
| `N_SLICES` | slices kept per development/test subject, chosen automatically through the knee |
| `N_TRAIN_SLICES` | slices kept per U-Net training subject (and for the development subject, as U-Net validation data) |
| `N_ACS` | fully sampled central k-space lines kept in every mask (also used to estimate coil maps) |
| `R_MAIN`, `R_CURVE` | acceleration factors: main comparison, and the wider quality-vs-acceleration curve |
| `SEEDS` | random masks are drawn with each seed, so results do not depend on one lucky mask |
| `PC_*`, `MC_*` | tuning grids (λ and iteration checkpoints) for per-coil and multi-coil FISTA |
| `ENERGY_FRAC`, `KEEP_CENTRAL*` | slice selection: slices whose signal energy is at least `ENERGY_FRAC` of the strongest one define the leg range, and the central part of that range is used |

In [ ]:
FORCE_RERUN = False
FORCE_DOWNLOAD = False
INPUT_ROOT = "/kaggle/input"

UUIDS = [
    "52c2fd53-d233-4444-8bfd-7c454240d314", "b65b9167-a0d0-4295-9dd5-74641b3dd7e6",
    "8ad53ab7-07f9-4864-98d0-dc43145ff588", "cc70c22f-1ddc-4a53-9503-e33d613de321",
    "280cf3f9-3b7e-4738-84e0-f72b21aa5266", "38b9a8e8-2779-4979-8602-5e8e5f902863",
    "54c077b2-7d68-4e77-b729-16afbccae9ac", "ec00945c-ad90-46b7-8c38-a69e9e801074",
    "dd969854-ec56-4ccc-b7ac-ff4cd7735095", "efa383b6-9446-438a-9901-1fe951653dbd",
    "8eff1229-8074-41fa-8b5e-441b501f10e3", "7a9f028c-8667-48aa-8e08-0acf3320c8d4",
    "ee2efe48-1e9d-480e-9364-e53db01532d4", "9a740e7b-8fc3-46f9-9f70-1b7bedec37e4",
    "530a812a-4870-4d01-9db4-772c853d693c", "1b197efe-9865-43be-ac24-f237c380513e",
    "226e710b-725b-4bec-840e-bf47be2b8a44", "2588bfa8-0c97-478c-aa5a-487cc88a590d",
    "b7d435a1-2421-48d2-946c-d1b3372f7c60", "d089cbe0-48b3-4ae2-9475-53ca89ee90fe",
]
BASE_URL = "http://mridata.org/download/"
HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/89.0.4389.82 Safari/537.36"}

TUNE_SUBJECT = 0
TEST_SUBJECTS = [4, 9, 14, 19]
UNAVAILABLE = {18}
TRAIN_SUBJECTS = [i for i in range(len(UUIDS)) if i != TUNE_SUBJECT and i not in TEST_SUBJECTS and i not in UNAVAILABLE]
N_SLICES, N_TRAIN_SLICES = 8, 40
ENERGY_FRAC, KEEP_CENTRAL, KEEP_CENTRAL_TRAIN = 0.5, 0.7, 0.85
TUNE_POS = [1, 4, 6]

N_ACS = 24
R_MAIN = [4, 8]
R_CURVE = [2, 3, 4, 6, 8]
SEEDS = [0, 1, 2]
PATTERNS = ["vd", "random", "equispaced"]
ESPIRIT = {"kw": 6, "thresh": 0.02, "crop": 0.0}

PC_LAMS, PC_CKPT = [1e-4, 3e-4, 1e-3], [25, 50, 100, 150]
MC_LAMS, MC_CKPT = [2.5e-4, 5e-4, 1e-3, 2e-3, 4e-3, 8e-3], [5, 10, 15, 20, 30, 50]

COIL_COUNTS, COIL_SUBSETS, COIL_R = [1, 2, 4, 8], 3, 8

MOTION_SLICES_PER_SUBJECT, MOTION_CASES_PER_SLICE = 2, 3
MOTION_MAX_SHIFT, MOTION_SEARCH = 6.0, 8

### Core functions
* **Fourier and coils:** centred orthonormal 2D FFT pair (`fft2c`, `ifft2c`) and root-sum-of-squares coil combination (`rss`).
* **Masks:** which ky lines are kept. Every mask keeps `N_ACS` central lines; the rest follow a pattern (`vd` = variable density, `random`, `equispaced`). `psf` is the inverse FFT of a mask.

In [ ]:
def fft2c(x):
    return sfft.fftshift(sfft.fft2(sfft.ifftshift(x, axes=(-2, -1)), norm="ortho", workers=-1), axes=(-2, -1))

def ifft2c(x):
    return sfft.fftshift(sfft.ifft2(sfft.ifftshift(x, axes=(-2, -1)), norm="ortho", workers=-1), axes=(-2, -1))

def rss(coil_imgs):
    return np.sqrt((np.abs(coil_imgs) ** 2).sum(axis=0))

def make_mask(n, R, pattern="vd", seed=0, n_acs=N_ACS):
    c = n // 2
    mask = np.zeros(n, bool)
    mask[c - n_acs // 2: c - n_acs // 2 + n_acs] = True
    outside = np.flatnonzero(~mask)
    n_extra = max(int(round(n / R)) - int(mask.sum()), 0)
    rng = np.random.default_rng(seed)
    if pattern == "equispaced":
        pick = outside[np.round(np.linspace(0, len(outside) - 1, n_extra)).astype(int)]
    elif pattern == "random":
        pick = rng.choice(outside, n_extra, replace=False)
    elif pattern == "vd":
        p = (1 - np.abs(outside - c) / c) ** 3
        pick = rng.choice(outside, n_extra, replace=False, p=p / p.sum())
    else:
        raise ValueError(f"unknown pattern {pattern}")
    mask[pick] = True
    return mask

def undersample(ks, mask):
    return ks * mask[None, :, None]

def psf(mask):
    p = np.abs(np.fft.fftshift(np.fft.ifft(np.fft.ifftshift(mask.astype(float)))))
    return p / p.max()

* **Wavelets and FISTA:** orthonormal `db4` wavelet (4 levels) applied to real and imaginary parts; soft-thresholding of all detail coefficients (the coarsest band is left untouched); a generic FISTA loop.
* **Reconstructions:** `recon_zf` (zero-filled), `recon_pc` (per-coil FISTA, then RSS), `recon_mc` (one image seen through coil sensitivity maps).

In [ ]:
WAVELET, LEVEL = "db4", 4

@lru_cache(maxsize=None)
def approx_shape(ny, nx):
    return pywt.wavedec2(np.zeros((ny, nx), np.float32), WAVELET, mode="periodization", level=LEVEL)[0].shape

def wt(x):
    ri = np.stack([x.real, x.imag])
    coeffs = pywt.wavedec2(ri, WAVELET, mode="periodization", level=LEVEL, axes=(-2, -1))
    arr, sl = pywt.coeffs_to_array(coeffs, axes=(-2, -1))
    return arr[0] + 1j * arr[1], sl

def iwt(c, sl):
    ri = np.stack([c.real, c.imag])
    out = pywt.waverec2(pywt.array_to_coeffs(ri, sl, output_format="wavedec2"),
                        WAVELET, mode="periodization", axes=(-2, -1))
    return out[0] + 1j * out[1]

def prox(x, lam):
    c, sl = wt(x)
    a, b = approx_shape(*x.shape[-2:])
    coarse = c[..., :a, :b].copy()
    c = c * np.maximum(1 - lam / np.maximum(np.abs(c), 1e-12), 0)
    c[..., :a, :b] = coarse
    return iwt(c, sl).astype(x.dtype)

def fista(A, AH, y, lam, n_iter, out_fn, checkpoints=()):
    x = AH(y)
    z, t = x.copy(), 1.0
    snaps = {}
    for it in range(1, n_iter + 1):
        xn = prox(z - AH(A(z) - y), lam)
        tn = (1 + math.sqrt(1 + 4 * t * t)) / 2
        z = xn + ((t - 1) / tn) * (xn - x)
        x, t = xn, tn
        if it in checkpoints:
            snaps[it] = out_fn(x)
    return out_fn(x), snaps

def recon_zf(y):
    return rss(ifft2c(y))

def recon_pc(y, mask, lam, n_iter, checkpoints=()):
    M = mask[None, :, None]
    return fista(lambda x: M * fft2c(x), lambda r: ifft2c(M * r), y, lam, n_iter, rss, checkpoints)

def recon_mc(y, mask, maps, lam, n_iter, checkpoints=()):
    M = mask[None, :, None]
    return fista(lambda x: M * fft2c(maps * x[None]),
                 lambda r: (maps.conj() * ifft2c(M * r)).sum(axis=0),
                 y, lam, n_iter, np.abs, checkpoints)

* **Coil sensitivity maps**, both estimated only from the fully sampled central `N_ACS × N_ACS` block:
  * `espirit_maps`: ESPIRiT. A calibration matrix of k-space patches is built, its dominant singular vectors are taken as k-space kernels, and the maps are the per-pixel dominant eigenvectors of the resulting image-domain operator.
  * `lowres_maps`: simple low-resolution coil images divided by their RSS (a cruder reference method).

  Both are normalised so that Σ|S_c|² ≤ 1, which keeps a FISTA step size of 1 valid.

In [ ]:
def calib_region(ks, n_cal):
    _, ny, nx = ks.shape
    return (slice(None), slice(ny // 2 - n_cal // 2, ny // 2 - n_cal // 2 + n_cal),
            slice(nx // 2 - n_cal // 2, nx // 2 - n_cal // 2 + n_cal))

def espirit_maps(ks, n_cal=N_ACS, kw=6, thresh=0.02, crop=0.0):
    nc, ny, nx = ks.shape
    cal = ks[calib_region(ks, n_cal)].astype(np.complex128)
    blocks = np.lib.stride_tricks.sliding_window_view(cal, (kw, kw), axis=(1, 2))
    A = blocks.reshape(nc, -1, kw * kw).transpose(1, 0, 2).reshape(-1, nc * kw * kw)
    _, s, vh = np.linalg.svd(A, full_matrices=False)
    kernels = vh[s > thresh * s.max()].reshape(-1, nc, kw, kw)
    AHA = np.zeros((ny * nx, nc, nc), np.complex64)
    y0, x0 = ny // 2 - kw // 2, nx // 2 - kw // 2
    for i in range(0, len(kernels), 32):
        ker = kernels[i:i + 32]
        pad = np.zeros((len(ker), nc, ny, nx), np.complex64)
        pad[:, :, y0:y0 + kw, x0:x0 + kw] = ker
        img = np.ascontiguousarray(ifft2c(pad).reshape(len(ker), nc, -1).transpose(2, 1, 0))
        AHA += img @ img.conj().transpose(0, 2, 1)
    AHA *= ny * nx / kw ** 2
    w, v = np.linalg.eigh(AHA)
    maps = v[:, :, -1].T.reshape(nc, ny, nx)
    eig = w[:, -1].reshape(ny, nx)
    maps = maps * np.exp(-1j * np.angle(maps.sum(axis=0)))[None]
    maps = maps * (eig > crop)[None]
    return maps.astype(np.complex64), eig

def lowres_maps(ks, n_cal=N_ACS):
    region = calib_region(ks, n_cal)
    win = np.outer(np.hanning(n_cal + 2)[1:-1], np.hanning(n_cal + 2)[1:-1])
    cal = np.zeros_like(ks)
    cal[region] = ks[region] * win
    low = ifft2c(cal)
    norm = rss(low)
    return (low / np.maximum(norm, 0.02 * norm.max())).astype(np.complex64)

* **Metrics** (all images are normalised so the ground truth peaks at 1):
  * **PSNR** (dB, higher is better): 10·log₁₀(1 / mean squared error). +1 dB ≈ 21 % less squared error, +3 dB ≈ half.
  * **SSIM** (0–1, higher is better): local similarity of brightness, contrast and structure. Reported over the whole image (`ssim`) and only inside the knee (`ssim_fg`), because the background contains only scanner noise.
  * **NMSE** (lower is better): squared error divided by the energy of the ground truth.
* **Utilities:** result caching to JSON, and plotting helpers.

In [ ]:
def foreground_mask(ref):
    sm = ndimage.gaussian_filter(ref, 2)
    m = sm > 0.5 * threshold_otsu(sm)
    m = ndimage.binary_closing(m, iterations=3)
    m = ndimage.binary_fill_holes(m)
    lab, n = ndimage.label(m)
    if n > 1:
        m = lab == (np.argmax(np.bincount(lab.ravel())[1:]) + 1)
    return m

def metrics(img, ref, fg):
    img, ref = np.asarray(img, np.float64), np.asarray(ref, np.float64)
    err = img - ref
    ssim, smap = structural_similarity(ref, img, data_range=1.0, full=True)
    return {"psnr": float(10 * np.log10(1.0 / np.mean(err ** 2))), "ssim": float(ssim),
            "ssim_fg": float(smap[fg].mean()), "nmse": float(np.sum(err ** 2) / np.sum(ref ** 2))}

def _jsonable(o):
    return o.item() if hasattr(o, "item") else str(o)

def cached(name, fn):
    path = os.path.join(RES_DIR, name)
    if os.path.exists(path) and not FORCE_RERUN:
        with open(path) as f:
            return json.load(f)
    out = fn()
    with open(path, "w") as f:
        json.dump(out, f, default=_jsonable)
    return out

def show(ax, img, title=None, vmin=0, vmax=1, cmap="gray"):
    ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
    if title:
        ax.set_title(title)
    ax.axis("off")

def show_kspace(ax, k, title=None):
    lk = np.log10(np.abs(k) + 1e-9)
    ax.imshow(lk, cmap="gray", vmin=np.percentile(lk, 1), vmax=lk.max())
    if title:
        ax.set_title(title)
    ax.axis("off")

def save(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name))
    plt.show()

def fmt_ms(s, digits):
    return f"{s.mean():.{digits}f} ± {s.std():.{digits}f}"

## Phase 1 · Data
**Dataset:** *Stanford Fully Sampled 3D FSE Knees* (mridata.org): raw multi-coil k-space of 20 healthy volunteers, stored in the ISMRMRD/HDF5 format. Each file holds one entry per acquired k-space line: a header (which ky line and which slice it belongs to) and the complex samples of that line for all coils.

For every subject:
1. **Download** to temporary disk and check the byte count against the server's.
2. **Read and assemble** `k[slice, coil, ky, kx]`, and check that every (slice, ky) line was acquired exactly once, i.e. the data is **fully sampled**. Its full reconstruction is therefore a true **ground truth**.
3. **Select slices** through the knee automatically. In this dataset the slice axis is already in image space, so each slice is an independent 2D k-space, and its energy equals its image energy. Slices whose smoothed energy is at least `ENERGY_FRAC` of the strongest define the extent of the leg; the central part of that range is kept and slices are taken evenly from it. Empty slices outside the leg contain only noise and have very low energy, so they are never selected.
4. **Save** and **delete** the raw file, and **log** where the data came from.
   * development and test subjects → `data/` (`N_SLICES` slices each, used by every experiment in this notebook);
   * training subjects and the development subject → `unet_data/` (`N_TRAIN_SLICES` slices each, raw coil k-space). Notebook 2 applies masks to these on the fly, so the U-Net sees the same kind of undersampling as the methods here. Test subjects are never written to `unet_data/`.

Files are written under a temporary name and renamed when complete, so an interruption never leaves a half-written file. A subject whose file cannot be read is reported as skipped and the notebook continues.

**Resuming:** if the output of an earlier version of this notebook is attached as an input (*Add Input → Notebook Output Files*), its finished subject files are copied in first and those subjects are not downloaded again.

The download-and-prepare step for one subject is a small standalone Python script (`code/prep_subject.py`, written by the next cell). The notebook runs it once per subject as a separate program: memory is fully released after every subject, a crash in one subject cannot stop the notebook, and the progress appears as a few plain lines per subject.

In [ ]:
PREP_SCRIPT = '''
import os, sys, json, time, gc
import numpy as np, h5py, requests

def log(msg):
    print(msg, flush=True)

def download(url, path, headers, retries=3):
    for attempt in range(1, retries + 1):
        try:
            t0 = time.time()
            r = requests.get(url, headers=headers, stream=True, timeout=60)
            r.raise_for_status()
            total = int(r.headers.get("Content-Length", 0))
            done, mark = 0, 0.25
            with open(path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
                    done += len(chunk)
                    if total and done / total >= mark:
                        log(f"  download {mark:.0%}: {done / 1e9:.2f} GB at {done / 1e6 / (time.time() - t0):.1f} MB/s")
                        mark += 0.25
            if total and done != total:
                raise IOError(f"incomplete download: {done} of {total} bytes")
            return done, r.url
        except Exception as e:
            log(f"  attempt {attempt} failed: {e}")
            time.sleep(5)
    raise RuntimeError("download failed")

def read_raw(path, batch=4096):
    with h5py.File(path, "r") as f:
        dset = f["dataset/data"]
        head = dset["head"]
        samples, coils = np.unique(head["number_of_samples"]), np.unique(head["active_channels"])
        if len(samples) != 1 or len(coils) != 1:
            raise ValueError("lines differ in length or coil count")
        ns, nc = int(samples[0]), int(coils[0])
        ky = head["idx"]["kspace_encode_step_1"].astype(np.int64)
        sl = head["idx"]["slice"].astype(np.int64)
        n_ky, n_sl = int(ky.max()) + 1, int(sl.max()) + 1
        size_gb = n_sl * nc * n_ky * ns * 8 / 1e9
        if size_gb > 4 or len(ky) != n_sl * n_ky:
            raise ValueError(f"unexpected layout: {n_sl} slices x {n_ky} lines x {nc} coils x {ns} samples "
                             f"({size_gb:.1f} GB), {len(ky)} acquisitions")
        k = np.zeros((n_sl, nc, n_ky, ns), np.complex64)
        count = np.zeros((n_sl, n_ky), np.int32)
        for start in range(0, len(ky), batch):
            data = np.stack(dset[start:start + batch]["data"]).view(np.complex64).reshape(-1, nc, ns)
            s_b, y_b = sl[start:start + batch], ky[start:start + batch]
            k[s_b, :, y_b, :] = data
            np.add.at(count, (s_b, y_b), 1)
    info = {"acquisitions": int(len(ky)), "readout_samples": ns, "coils": nc, "ky_lines": n_ky,
            "slices": n_sl, "fully_sampled_once": bool((count == 1).all())}
    return k, info

def select_slices(energy, n, keep, energy_frac):
    smooth = np.convolve(energy, np.ones(9) / 9, mode="same")
    above = np.flatnonzero(smooth >= energy_frac * smooth.max())
    lo, hi = above.min(), above.max()
    margin = (hi - lo) * (1 - keep) / 2
    return np.unique(np.round(np.linspace(lo + margin, hi - margin, n)).astype(int))

cfg = json.load(open(sys.argv[1]))
os.makedirs(cfg["temp"], exist_ok=True)
path = os.path.join(cfg["temp"], cfg["uuid"] + ".h5")
try:
    size, served_from = download(cfg["url"], path, cfg["headers"])
    k, info = read_raw(path)
    log(f"  read: {info}")
    if not info["fully_sampled_once"]:
        raise ValueError("not every (slice, ky) line was acquired exactly once")
    energy = np.array([np.vdot(s, s).real for s in k])
    entry = {"subject": cfg["idx"], "uuid": cfg["uuid"], "url": cfg["url"], "served_from": served_from,
             "bytes": size, **info}
    for out, n, keep, key in cfg["targets"]:
        chosen = select_slices(energy, n, keep, cfg["energy_frac"])
        with open(out + ".part", "wb") as f:
            np.savez(f, kspace=k[chosen], slices=chosen, energy=energy)
        os.replace(out + ".part", out)
        entry[key] = chosen.tolist()
    with open(cfg["result"], "w") as f:
        json.dump(entry, f)
finally:
    if os.path.exists(path):
        os.remove(path)
'''
PREP_SCRIPT_PATH = os.path.join(CODE_DIR, "prep_subject.py")
with open(PREP_SCRIPT_PATH, "w") as f:
    f.write(PREP_SCRIPT)

EVAL_SUBJECTS = [TUNE_SUBJECT] + TEST_SUBJECTS
UNET_SUBJECTS = TRAIN_SUBJECTS + [TUNE_SUBJECT]

def prepare_subject(idx):
    targets = []
    if idx in EVAL_SUBJECTS:
        targets.append([os.path.join(DATA_DIR, f"subject{idx:02d}.npz"), N_SLICES, KEEP_CENTRAL, "eval_slices"])
    if idx in UNET_SUBJECTS:
        targets.append([os.path.join(UNET_DIR, f"subject{idx:02d}.npz"), N_TRAIN_SLICES, KEEP_CENTRAL_TRAIN, "unet_slices"])
    if not FORCE_DOWNLOAD and all(os.path.exists(t[0]) for t in targets):
        return None
    job, result = os.path.join(CODE_DIR, f"job{idx:02d}.json"), os.path.join(CODE_DIR, f"result{idx:02d}.json")
    if os.path.exists(result):
        os.remove(result)
    with open(job, "w") as f:
        json.dump({"idx": idx, "uuid": UUIDS[idx], "url": BASE_URL + UUIDS[idx], "headers": HEADERS, "temp": TEMP,
                   "targets": targets, "energy_frac": ENERGY_FRAC, "result": result}, f)
    proc = subprocess.Popen([sys.executable, PREP_SCRIPT_PATH, job], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    tail = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line.strip()])[-3:]
    if proc.wait() != 0 or not os.path.exists(result):
        raise RuntimeError(f"preparation stopped (exit code {proc.returncode}): {' | '.join(tail)}")
    entry = json.load(open(result))
    entry["downloaded_on"] = str(datetime.date.today())
    return entry

In [ ]:
log_path = os.path.join(RES_DIR, "data_log.json")
data_log = json.load(open(log_path)) if os.path.exists(log_path) else {}

def resume_from_inputs():
    copied = 0
    if not os.path.isdir(INPUT_ROOT):
        return copied
    for dirpath, _, files in os.walk(INPUT_ROOT):
        name = os.path.basename(dirpath)
        if name == "results" and "data_log.json" in files:
            for key, entry in json.load(open(os.path.join(dirpath, "data_log.json"))).items():
                data_log.setdefault(key, entry)
        if name not in ("data", "unet_data"):
            continue
        dest = DATA_DIR if name == "data" else UNET_DIR
        for f in files:
            if f.startswith("subject") and f.endswith(".npz") and not os.path.exists(os.path.join(dest, f)):
                shutil.copy(os.path.join(dirpath, f), os.path.join(dest, f))
                copied += 1
    return copied

print(f"subject files reused from attached inputs: {resume_from_inputs()}")

ready = []
for idx in sorted(set(EVAL_SUBJECTS) | set(UNET_SUBJECTS)):
    t0 = time.time()
    print(f"subject {idx}:", flush=True)
    try:
        entry = prepare_subject(idx)
        if entry:
            data_log[str(idx)] = entry
        with open(log_path, "w") as f:
            json.dump(data_log, f, indent=1)
        ready.append(idx)
        print(f"  ready ({time.time() - t0:.0f} s)", flush=True)
    except Exception as e:
        print(f"  skipped: {e}", flush=True)
    finally:
        for f in os.listdir(TEMP):
            os.remove(os.path.join(TEMP, f))

SUBJECTS = [s for s in EVAL_SUBJECTS if s in ready]
TEST = [s for s in TEST_SUBJECTS if s in ready]
TRAIN_OK = [s for s in TRAIN_SUBJECTS if s in ready]
assert TUNE_SUBJECT in ready and TEST, "the development subject and at least one test subject are required"

role = {TUNE_SUBJECT: "development", **{s: "test" for s in TEST_SUBJECTS}, **{s: "training" for s in TRAIN_SUBJECTS}}
cols = ["subject", "role", "coils", "readout_samples", "ky_lines", "slices", "fully_sampled_once", "eval_slices", "unet_slices"]
table0 = pd.DataFrame([{**data_log[str(s)], "role": role[s]} for s in ready if str(s) in data_log]).reindex(columns=cols)
table0.to_csv(os.path.join(RES_DIR, "table0_data.csv"), index=False)
display(table0)
print(f"development: {TUNE_SUBJECT} | test: {TEST} | U-Net training: {TRAIN_OK}")

project_config = {"development_subject": TUNE_SUBJECT, "test_subjects": TEST, "train_subjects": TRAIN_OK,
                  "n_acs": N_ACS, "r_main": R_MAIN, "r_curve": R_CURVE, "seeds": SEEDS, "patterns": PATTERNS,
                  "mask_rule": "N_ACS central lines + round(n/R) total; vd: p ∝ (1 - |k - c| / c)^3 over the rest; "
                               "numpy default_rng(seed)",
                  "normalisation": "k-space divided by the max of its fully sampled RSS image (ground truth peaks at 1)",
                  "eval_files": "data/subjectXX.npz", "unet_files": "unet_data/subjectXX.npz",
                  "array_layout": "kspace[slice, coil, ky, kx], complex64, k-space centre at index n // 2"}
with open(os.path.join(RES_DIR, "project_config.json"), "w") as f:
    json.dump(project_config, f, indent=1)

In [ ]:
_store = {}

def subject_data(idx):
    if idx not in _store:
        d = np.load(os.path.join(DATA_DIR, f"subject{idx:02d}.npz"))
        _store[idx] = {"kspace": d["kspace"], "slices": d["slices"], "energy": d["energy"]}
    return _store[idx]

def n_saved(idx):
    return len(subject_data(idx)["slices"])

def get_slice(idx, pos, coils=None):
    ks = subject_data(idx)["kspace"][pos]
    if coils is not None:
        ks = ks[list(coils)]
    ref = rss(ifft2c(ks))
    scale = ref.max()
    return (ks / scale).astype(np.complex64), (ref / scale).astype(np.float32)

ncols = max(n_saved(s) for s in SUBJECTS) + 1
fig, axes = plt.subplots(len(SUBJECTS), ncols, figsize=(1.9 * ncols, 2.1 * len(SUBJECTS)), squeeze=False)
for r, s in enumerate(SUBJECTS):
    d = subject_data(s)
    e = d["energy"] / d["energy"].max()
    axes[r, 0].plot(e, lw=1)
    axes[r, 0].plot(d["slices"], e[d["slices"]], "o", ms=3)
    axes[r, 0].set_title(f"subject {s} ({role[s]}): energy per slice")
    axes[r, 0].set_xlabel("slice index")
    for c in range(1, ncols):
        if c - 1 < n_saved(s):
            show(axes[r, c], get_slice(s, c - 1)[1], f"slice {d['slices'][c - 1]}")
        else:
            axes[r, c].axis("off")
plt.tight_layout()
save(fig, "fig00_selected_slices.png")

In [ ]:
unet_list = [s for s in UNET_SUBJECTS if s in ready]
per_row = 4
rows_n = math.ceil(len(unet_list) / per_row)
fig, axes = plt.subplots(rows_n, 2 * per_row, figsize=(2.0 * 2 * per_row, 2.2 * rows_n), squeeze=False)
for a in axes.ravel():
    a.axis("off")
sizes = []
for i, s in enumerate(unet_list):
    path = os.path.join(UNET_DIR, f"subject{s:02d}.npz")
    d = np.load(path)
    sizes.append(os.path.getsize(path))
    for j, pos in enumerate([0, len(d["slices"]) - 1]):
        img = rss(ifft2c(d["kspace"][pos]))
        show(axes[i // per_row, 2 * (i % per_row) + j], img / img.max(), f"subj {s}, slice {d['slices'][pos]}")
    del d
plt.tight_layout()
save(fig, "fig00b_unet_slice_range.png")
print(f"U-Net data: {len(unet_list)} subjects, {sum(sizes) / 1e9:.2f} GB in {UNET_DIR}")

**Check:** the first figure shows, for each development/test subject, the signal energy across all slices (selected slices marked) and the selected slices. The second shows the first and last selected slice of every U-Net subject, the ones most likely to fall outside the knee. All thumbnails should show knee anatomy. If a selection contains empty or skin-only slices, change `ENERGY_FRAC` / `KEEP_CENTRAL` / `KEEP_CENTRAL_TRAIN`, set `FORCE_DOWNLOAD = True` and `FORCE_RERUN = True`, and run the notebook again.

## Phase 2 · k-space anatomy
The scanner measures **k-space**: the 2D Fourier transform of the image. A point near the centre describes broad, slow variations (overall brightness and contrast); a point far from the centre describes fine, fast variations (edges and texture). The inverse FFT turns k-space into the image.

Each of the coils sees the knee through its own **sensitivity**: bright close to the coil, dim far from it. The coil images are combined by **root-sum-of-squares (RSS)**, and the RSS of the fully sampled data is the ground truth used everywhere below.

*Figure 1* uses the middle saved slice of the tuning subject.

In [ ]:
MID = n_saved(TUNE_SUBJECT) // 2
ks, ref = get_slice(TUNE_SUBJECT, MID)
nc, ny, nx = ks.shape
coil_imgs = ifft2c(ks)
strong = int(np.argmax((np.abs(ks) ** 2).sum(axis=(1, 2))))
coil_pick = np.linspace(0, nc - 1, 4).astype(int)

fig, ax = plt.subplots(2, 4, figsize=(15, 7.6))
show_kspace(ax[0, 0], ks[strong], f"k-space, coil {strong} (log magnitude)")
show_kspace(ax[0, 1], np.sqrt((np.abs(ks) ** 2).sum(axis=0)), "k-space, all coils (log magnitude)")
show(ax[0, 2], ref, "RSS image (ground truth)")
ax[0, 3].semilogy(np.abs(ks[strong][:, nx // 2]))
ax[0, 3].set_title(f"coil {strong}: |k| along ky at the kx centre")
ax[0, 3].set_xlabel("ky line")
for a, c in zip(ax[1], coil_pick):
    show(a, np.abs(coil_imgs[c]), f"image from coil {c} only", vmax=np.abs(coil_imgs).max())
plt.tight_layout()
save(fig, "fig01_kspace_anatomy.png")

mag = np.abs(ks).ravel()
print(f"slice {subject_data(TUNE_SUBJECT)['slices'][MID]} | {nc} coils | {ny} ky lines x {nx} readout samples")
print(f"k-space dynamic range (max / median magnitude): {mag.max() / np.median(mag):.0f}x")

**Reading Figure 1:** k-space is dominated by its centre, so it is shown on a log scale; the profile along ky drops by orders of magnitude from the centre outward. Bright streaks through the centre come from sharp edges in the image (an edge puts energy along the line perpendicular to it). Each single-coil image is bright near its coil and dark elsewhere; the RSS image combines them into one evenly lit image.

## Phase 3 · Where the information lives
Keep only a disk of radius *r* around the k-space centre (**low frequencies**), or only everything outside it (**high frequencies**), and reconstruct. The table reports what fraction of k-space points and of signal energy each disk holds.

A hard cut-off in k-space multiplies k-space by a disk, which is the same as convolving the image with the disk's transform (a ringing, sinc-like pattern). This produces **Gibbs ringing** next to sharp edges.

In [ ]:
fg = foreground_mask(ref)
yy, xx = np.ogrid[:ny, :nx]
dist = np.hypot(yy - ny // 2, xx - nx // 2)
energy_total = (np.abs(ks) ** 2).sum()
radii = [6, 12, 24, 48]
rows, pairs = [], []
for r in radii:
    low = dist <= r
    img_low, img_high = recon_zf(ks * low[None]), recon_zf(ks * (~low)[None])
    rows.append({"radius": r, "% of k-space points": 100 * low.mean(),
                 "% of signal energy": 100 * (np.abs(ks * low[None]) ** 2).sum() / energy_total,
                 "PSNR centre only (dB)": metrics(img_low, ref, fg)["psnr"],
                 "PSNR outside only (dB)": metrics(img_high, ref, fg)["psnr"]})
    pairs.append((img_low, img_high))
table1 = pd.DataFrame(rows)
table1.to_csv(os.path.join(RES_DIR, "table1_centre_vs_outside.csv"), index=False)
display(table1.round(2))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].bar([str(r) for r in radii], table1["% of signal energy"], color="C0")
ax[0].bar_label(ax[0].containers[0], fmt="%.0f%%")
ax[0].set_title("signal energy inside the central disk"); ax[0].set_xlabel("disk radius (k-space samples)"); ax[0].set_ylabel("%")
ax[1].plot(radii, table1["PSNR centre only (dB)"], "o-", label="centre only")
ax[1].plot(radii, table1["PSNR outside only (dB)"], "s--", label="outside only")
ax[1].set_title("PSNR of the partial reconstructions"); ax[1].set_xlabel("disk radius"); ax[1].set_ylabel("dB"); ax[1].legend()
plt.tight_layout()
save(fig, "fig02b_centre_vs_outside_chart.png")

fig, ax = plt.subplots(2, len(radii) + 1, figsize=(3.4 * (len(radii) + 1), 7))
show(ax[0, 0], ref, "ground truth")
ax[1, 0].axis("off")
for j, (r, (lo_img, hi_img)) in enumerate(zip(radii, pairs), start=1):
    show(ax[0, j], lo_img, f"centre only, r = {r}")
    show(ax[1, j], hi_img, f"outside only, r = {r} (display x3)", vmax=1 / 3)
plt.tight_layout()
save(fig, "fig02_centre_vs_outside.png")

**Reading Table 1 / Figure 2:** a tiny central disk holds most of the signal energy and already gives the overall contrast; growing the disk adds sharpness. The outside alone contains only edges and texture. Its PSNR is low because the broad brightness is missing, not because it carries no information: energy and visual detail are different things. This is why every mask below keeps a fully sampled centre.

## Phase 4 · Undersampling and aliasing
Along the readout direction (**kx**) a full line is recorded in a few milliseconds, but every **phase-encoding line (ky)** costs a separate measurement, so scan time is proportional to the number of ky lines. A faster scan therefore means **keeping fewer ky lines**.

* **Acceleration factor R** = total ky lines / kept lines. R = 4 keeps a quarter of the lines; R = 8 keeps an eighth.
* Every mask keeps the central `N_ACS` lines (the **autocalibration** or ACS region). The remaining lines are chosen *equispaced*, *random*, or *variable density* (random, but denser near the centre).
* This is **retrospective undersampling**: lines are deleted from fully sampled data, so the ground truth is still known.
* **Zero-filled reconstruction** puts zeros in the missing lines and applies the inverse FFT; it is the baseline.
* Multiplying k-space by a mask convolves the image with the mask's **point spread function (PSF)**, the inverse FFT of the mask. A PSF with a few strong side peaks produces **coherent ghost copies**; a PSF with many small side lobes produces **incoherent, noise-like aliasing**.

In [ ]:
rows = []
for R in R_CURVE:
    m = make_mask(ny, R, "vd", SEEDS[0])
    rows.append({"nominal R": R, "lines kept": int(m.sum()), "of total": ny, "effective R": ny / m.sum(),
                 "ACS lines": N_ACS, "lines outside ACS": int(m.sum()) - N_ACS})
display(pd.DataFrame(rows))

In [ ]:
R_DEMO = 4
fig, ax = plt.subplots(4, 4, figsize=(15, 14), gridspec_kw={"height_ratios": [0.35, 1, 1, 1]})
ax[0, 0].axis("off"); show(ax[1, 0], ref, "ground truth"); ax[2, 0].axis("off"); ax[3, 0].axis("off")
for j, pattern in enumerate(["equispaced", "random", "vd"], start=1):
    m = make_mask(ny, R_DEMO, pattern, SEEDS[0])
    zf = recon_zf(undersample(ks, m))
    s = metrics(zf, ref, fg)
    ax[0, j].imshow(np.tile(m, (12, 1)), cmap="gray", aspect="auto"); ax[0, j].axis("off")
    ax[0, j].set_title(f"{pattern} mask: {m.sum()} of {ny} lines")
    show(ax[1, j], zf, f"zero-filled  PSNR {s['psnr']:.1f} dB  SSIM(knee) {s['ssim_fg']:.3f}")
    show(ax[2, j], np.abs(zf - ref), "absolute error", vmax=0.3, cmap="inferno")
    ax[3, j].plot(psf(m)); ax[3, j].set_ylim(0, 0.6); ax[3, j].set_title("PSF of the mask (peak clipped)")
    ax[3, j].set_xlabel("image row")
plt.tight_layout()
save(fig, f"fig03_masks_R{R_DEMO}.png")

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(14, 9))
show(ax[0, 0], ref, "ground truth"); ax[1, 0].axis("off")
for j, pattern in enumerate(["equispaced", "random"], start=1):
    m = make_mask(ny, R_DEMO, pattern, SEEDS[0], n_acs=0)
    zf = recon_zf(undersample(ks, m))
    show(ax[0, j], zf, f"{pattern}, no ACS block  PSNR {metrics(zf, ref, fg)['psnr']:.1f} dB")
    ax[1, j].plot(psf(m)); ax[1, j].set_ylim(0, 0.6); ax[1, j].set_title(f"PSF of the {pattern} mask (peak clipped)")
    ax[1, j].set_xlabel("image row")
plt.tight_layout()
save(fig, f"fig04_psf_without_acs_R{R_DEMO}.png")

**Reading Figures 3–4:** with the central block kept, all patterns mainly lose fine detail (blur along the vertical, ky, direction), because the energy-heavy centre is intact. Without the central block the difference between patterns appears clearly: the equispaced PSF has a few tall side peaks spaced by about N/R lines, giving shifted ghost copies of the knee, while the random PSF has many small lobes, giving a diffuse haze. Compressed sensing (next phase) relies on this second kind: noise-like artifacts can be separated from a sparse image, whereas ghost copies look like anatomy.

## Phase 5 · Reconstruction methods
**Compressed sensing.** Infinitely many images agree with the measured lines, so an assumption about real images is added: natural images are **sparse** in a wavelet basis (a few large coefficients, many near zero), while aliasing artifacts spread energy over many coefficients. The reconstruction solves

$$\min_x \; \tfrac12 \lVert M F x - y \rVert_2^2 + \lambda \lVert W x \rVert_1$$

where *F* is the FFT, *M* the mask, *y* the measured lines, *W* the wavelet transform and λ the balance between agreeing with the data and being sparse.

**FISTA** solves it by repeating: (1) a data-consistency step that corrects the image so its k-space agrees with the measured lines, (2) soft-thresholding of the wavelet coefficients (small ones become zero), plus (3) a momentum step that extrapolates from the last two iterates, which makes it converge much faster than plain iterative shrinkage.

**Two models of the coils:**
* **per-coil:** each coil image is reconstructed separately, then combined with RSS. Simple, but it ignores that all coils see the same knee.
* **multi-coil:** one image *x* is reconstructed, and coil *c* is modelled as measuring *M F (S_c x)*, where *S_c* is its sensitivity map (the SENSE model). The coils then act as extra measurements of the same object.

In [ ]:
maps_es, eig_es = espirit_maps(ks, **ESPIRIT)
x_comb = (maps_es.conj() * coil_imgs).sum(axis=0).astype(np.complex128)
c, sl = wt(x_comb)
print(f"wavelet round-trip relative error: {np.abs(iwt(c, sl) - x_comb).max() / np.abs(x_comb).max():.1e}")
print(f"energy ratio ||Wx|| / ||x||: {np.linalg.norm(c) / np.linalg.norm(x_comb):.6f} (1 = orthonormal)")

def n_for(a, frac):
    e = np.sort(np.abs(a).ravel() ** 2)[::-1]
    return int(np.searchsorted(np.cumsum(e) / e.sum(), frac)) + 1

rows = [{"energy kept": f"{f:.0%}", "pixels needed": n_for(x_comb, f), "wavelet coefficients needed": n_for(c, f),
         "ratio": n_for(x_comb, f) / n_for(c, f)} for f in (0.90, 0.95, 0.99)]
display(pd.DataFrame(rows))

**Sparsity check:** the wavelet transform must be orthonormal (energy ratio 1, round-trip error near zero) for soft-thresholding to be the exact shrinkage step. The table shows how many numbers hold 90 / 95 / 99 % of the image energy in pixels versus wavelets. A large ratio at 90–95 % means the anatomy is very sparse; the ratio shrinks at 99 % because the last percent is mostly scanner noise, which is not sparse.

In [ ]:
t0 = time.time()
maps_es, eig_es = espirit_maps(ks, **ESPIRIT)
t_es = time.time() - t0
maps_lr = lowres_maps(ks)
agree = np.abs((maps_es.conj() * maps_lr).sum(axis=0)) / (rss(maps_es) * rss(maps_lr) + 1e-12)
print(f"ESPIRiT time: {t_es:.1f} s | mean agreement of the two map sets inside the knee: {agree[fg].mean():.3f} (1 = same up to phase)")

fig, ax = plt.subplots(2, 5, figsize=(17, 7))
for j, c in enumerate(coil_pick):
    show(ax[0, j], np.abs(maps_es[c]), f"ESPIRiT |S| coil {c}")
    show(ax[1, j], np.abs(maps_lr[c]), f"low-res |S| coil {c}")
show(ax[0, 4], eig_es, "ESPIRiT eigenvalue map", vmax=1, cmap="viridis")
ax[1, 4].imshow(ref, cmap="gray", vmin=0, vmax=1)
ax[1, 4].contour(fg, levels=[0.5], colors="r", linewidths=1)
ax[1, 4].set_title("foreground mask used for ssim_fg"); ax[1, 4].axis("off")
plt.tight_layout()
save(fig, "fig05_sensitivity_maps.png")

**Reading Figure 5:** sensitivity maps should vary smoothly and cover the whole knee. The ESPIRiT eigenvalue is close to 1 where the coil model fits the data (inside the object) and lower in the noise-only background. Maps are not cropped (`crop = 0`), so no part of the anatomy can be cut out by a threshold. The red contour is the knee region used for `ssim_fg`; it should enclose the knee and exclude the background.

## Phase 6 · Tuning on a separate subject
FISTA has two knobs: **λ** (how strongly sparsity is enforced) and the **number of iterations** (quality can peak and then decline, so stopping early also acts as regularisation). They are chosen on a few slices of the tuning subject only, with a variable-density mask (first seed).

**Selection rule (fixed before looking at test data):** the setting with the highest mean PSNR, chosen **separately for every acceleration factor**: multi-coil FISTA at every R in `R_CURVE`, per-coil FISTA at `R_MAIN`. A lightly undersampled image needs much less regularisation than a heavily undersampled one, so one setting for all R would be unfair at the extremes. The low-res-map variant gets one setting over `R_MAIN`, for the map comparison in Phase 7. The chosen settings are then **frozen**.

In [ ]:
def run_tuning():
    rows = []
    positions = [p for p in TUNE_POS if p < n_saved(TUNE_SUBJECT)]
    for pos in tqdm(positions, desc="tuning slices"):
        ks_t, ref_t = get_slice(TUNE_SUBJECT, pos)
        fg_t = foreground_mask(ref_t)
        maps = {"multi-coil": espirit_maps(ks_t, **ESPIRIT)[0], "multi-coil (low-res maps)": lowres_maps(ks_t)}
        for R in R_CURVE:
            mask = make_mask(ks_t.shape[1], R, "vd", SEEDS[0])
            y = undersample(ks_t, mask)
            base = {"pos": pos, "R": R}
            rows.append({**base, "method": "zero-filled", "lam": 0.0, "iters": 0, **metrics(recon_zf(y), ref_t, fg_t)})
            if R in R_MAIN:
                for lam in PC_LAMS:
                    for it, img in recon_pc(y, mask, lam, max(PC_CKPT), PC_CKPT)[1].items():
                        rows.append({**base, "method": "per-coil", "lam": lam, "iters": it, **metrics(img, ref_t, fg_t)})
            for name, S in maps.items():
                if name != "multi-coil" and R not in R_MAIN:
                    continue
                for lam in MC_LAMS:
                    for it, img in recon_mc(y, mask, S, lam, max(MC_CKPT), MC_CKPT)[1].items():
                        rows.append({**base, "method": name, "lam": lam, "iters": it, **metrics(img, ref_t, fg_t)})
    return rows

t0 = time.time()
tune_df = pd.DataFrame(cached("tuning.json", run_tuning))
print(f"tuning results: {len(tune_df)} rows ({time.time() - t0:.0f} s)")

In [ ]:
def pick(method, Rs, lams, ckpts):
    sub = tune_df[(tune_df.method == method) & tune_df.R.isin(Rs)]
    score = sub.groupby(["lam", "iters"])["psnr"].mean()
    lam, it = score.idxmax()
    notes = [n for n, cond in (("λ at the edge of the grid", lam in (min(lams), max(lams))),
                               ("iterations at the edge of the grid", it == max(ckpts))) if cond]
    return float(lam), int(it), notes

PC_SET = {R: pick("per-coil", [R], PC_LAMS, PC_CKPT) for R in R_MAIN}
MC_SET = {R: pick("multi-coil", [R], MC_LAMS, MC_CKPT) for R in R_CURVE}
LR_SET = pick("multi-coil (low-res maps)", R_MAIN, MC_LAMS, MC_CKPT)

settings = [{"method": f"per-coil (R={R})", "lambda": v[0], "iterations": v[1], "note": "; ".join(v[2])} for R, v in PC_SET.items()]
settings += [{"method": f"multi-coil (R={R})", "lambda": v[0], "iterations": v[1], "note": "; ".join(v[2])} for R, v in MC_SET.items()]
settings += [{"method": "multi-coil, low-res maps (R in R_MAIN)", "lambda": LR_SET[0], "iterations": LR_SET[1], "note": "; ".join(LR_SET[2])}]
settings_df = pd.DataFrame(settings)
settings_df.to_csv(os.path.join(RES_DIR, "frozen_settings.csv"), index=False)
display(settings_df)

PC_SET = {R: v[:2] for R, v in PC_SET.items()}
MC_SET = {R: v[:2] for R, v in MC_SET.items()}
LR_SET = LR_SET[:2]

zf_ref = tune_df[tune_df.method == "zero-filled"].groupby("R")["psnr"].mean()
display(pd.DataFrame([{"R": R, "zero-filled": zf_ref[R], **{m: tune_df[(tune_df.method == m) & (tune_df.R == R)]
                       .groupby(["lam", "iters"])["psnr"].mean().max() for m in ("per-coil", "multi-coil") if R in tune_df[tune_df.method == m].R.values}}
                      for R in R_CURVE]).set_index("R").rename(columns=lambda c: f"best {c} PSNR" if c != "zero-filled" else "zero-filled PSNR"))
fig, ax = plt.subplots(1, 2 * len(R_MAIN), figsize=(5 * 2 * len(R_MAIN), 4))
for j, (method, R) in enumerate([(m, R) for m in ("per-coil", "multi-coil") for R in R_MAIN]):
    sub = tune_df[(tune_df.method == method) & (tune_df.R == R)]
    for lam, g in sub.groupby("lam"):
        g = g.groupby("iters")["psnr"].mean()
        ax[j].plot(g.index, g.values, "o-", ms=3, label=f"λ = {lam:g}")
    ax[j].axhline(zf_ref[R], color="gray", ls="--", label="zero-filled")
    ax[j].set_title(f"{method}, R = {R} (tuning slices)")
    ax[j].set_xlabel("iterations"); ax[j].set_ylabel("mean PSNR (dB)"); ax[j].legend(fontsize=8)
plt.tight_layout()
save(fig, "fig06_tuning_curves.png")

**Reading the tuning:** each curve is one λ; the dashed line is zero-filled. Curves that rise and then fall show that stopping early matters. If a chosen setting sits at the edge of its grid (see the *note* column), the true optimum may lie outside the grid; widening `PC_LAMS` / `MC_LAMS` / checkpoints and re-running Phase 6 and 7 (with `FORCE_RERUN = True`) is the remedy.

## Phase 7 · Held-out evaluation
The frozen settings are applied to every saved slice of every **test subject**, none of which were used for tuning.

* **Main comparison:** variable-density masks at R = 4 and 8, all seeds; zero-filled, per-coil FISTA and multi-coil FISTA.
* **Acceleration curve:** zero-filled and multi-coil at every R in `R_CURVE`.
* **Sampling pattern:** zero-filled and multi-coil with equispaced, random and variable-density masks at `R_MAIN`.
* **Map comparison:** multi-coil with low-res maps instead of ESPIRiT maps (first seed).

Coil maps are estimated once per slice from the ACS block, which every mask contains. Results are saved per subject, so an interrupted run resumes where it stopped.

In [ ]:
def eval_slice(subj, pos):
    ks_e, ref_e = get_slice(subj, pos)
    fg_e = foreground_mask(ref_e)
    S_es, S_lr = espirit_maps(ks_e, **ESPIRIT)[0], lowres_maps(ks_e)
    n = ks_e.shape[1]
    rows = []
    def add(R, seed, pattern, method, img):
        rows.append({"subject": subj, "pos": pos, "slice": int(subject_data(subj)["slices"][pos]), "R": R,
                     "seed": seed, "pattern": pattern, "method": method, **metrics(img, ref_e, fg_e)})
    for R in R_CURVE:
        for seed in SEEDS:
            for pattern in PATTERNS:
                if (pattern != "vd" and R not in R_MAIN) or (pattern == "equispaced" and seed != SEEDS[0]):
                    continue
                mask = make_mask(n, R, pattern, seed)
                y = undersample(ks_e, mask)
                add(R, seed, pattern, "zero-filled", recon_zf(y))
                add(R, seed, pattern, "multi-coil", recon_mc(y, mask, S_es, *MC_SET[R])[0])
                if pattern == "vd" and R in R_MAIN:
                    add(R, seed, pattern, "per-coil", recon_pc(y, mask, *PC_SET[R])[0])
                    if seed == SEEDS[0]:
                        add(R, seed, pattern, "multi-coil (low-res maps)", recon_mc(y, mask, S_lr, *LR_SET)[0])
    return rows

ks_b, _ = get_slice(TUNE_SUBJECT, MID)
m_b = make_mask(ny, R_MAIN[0], "vd", SEEDS[0])
y_b = undersample(ks_b, m_b)
t0 = time.time(); recon_pc(y_b, m_b, PC_SET[R_MAIN[0]][0], 5); t_pc = (time.time() - t0) / 5
t0 = time.time(); S_b = espirit_maps(ks_b, **ESPIRIT)[0]; t_map = time.time() - t0
t0 = time.time(); recon_mc(y_b, m_b, S_b, MC_SET[R_MAIN[0]][0], 5); t_mc = (time.time() - t0) / 5
mc_iters = sum(len(SEEDS) * MC_SET[R][1] for R in R_CURVE) + sum((len(SEEDS) + 1) * MC_SET[R][1] for R in R_MAIN)
n_runs = len(R_CURVE) * len(SEEDS) + len(R_MAIN) * (len(SEEDS) + 1)
per_slice = (t_map + mc_iters * t_mc + len(SEEDS) * sum(PC_SET[R][1] for R in R_MAIN) * t_pc
             + len(R_MAIN) * LR_SET[1] * t_mc + 3 * n_runs * 0.1)
n_slices_total = sum(n_saved(s) for s in TEST)
print(f"per iteration: per-coil {t_pc * 1000:.0f} ms, multi-coil {t_mc * 1000:.0f} ms | ESPIRiT {t_map:.1f} s")
print(f"estimated evaluation time: {per_slice * n_slices_total / 60:.0f} min for {n_slices_total} slices "
      "(only slices not already saved in results/ are computed)")

eval_rows = []
for subj in TEST:
    def run(subj=subj):
        rows = []
        for pos in tqdm(range(n_saved(subj)), desc=f"subject {subj}"):
            rows += eval_slice(subj, pos)
        return rows
    eval_rows += cached(f"eval_subject{subj:02d}.json", run)
ev = pd.DataFrame(eval_rows)
print(f"evaluation results: {len(ev)} rows")

In [ ]:
KEYS = ["subject", "pos", "R", "seed", "pattern"]
METRICS = {"psnr": 2, "ssim": 4, "ssim_fg": 4, "nmse": 4}

def paired(df, method, base="zero-filled"):
    a = df[df.method == method].set_index(KEYS)[list(METRICS)]
    b = df[df.method == base].set_index(KEYS)[list(METRICS)]
    a, b = a.align(b, join="inner")
    return a - b

NAMES = {"psnr": "PSNR (dB)", "ssim": "SSIM", "ssim_fg": "SSIM (knee)", "nmse": "NMSE"}
HIGHER = {"psnr": True, "ssim": True, "ssim_fg": True, "nmse": False}

def means_table(df, methods, bases):
    rows = []
    for R in sorted(df.R.unique()):
        dR = df[df.R == R]
        for m in methods:
            s = dR[dR.method == m]
            row = {"R": R, "method": m}
            row.update({NAMES[k]: round(s[k].mean(), d) for k, d in METRICS.items()})
            for b in bases:
                if m != b and m in dR.method.values and b in dR.method.values:
                    d = paired(dR, m, b)
                    row[f"PSNR better than {b}"] = f"{(d.psnr > 0).sum()} of {len(d)}"
            rows.append(row)
    return pd.DataFrame(rows).fillna("")

def bar_chart(df, methods, fname, title):
    Rs = sorted(df.R.unique())
    x, w = np.arange(len(Rs)), 0.8 / len(methods)
    fig, ax = plt.subplots(1, 4, figsize=(20, 4.4))
    for a, k in zip(ax, ["psnr", "ssim", "ssim_fg", "nmse"]):
        for i, m in enumerate(methods):
            vals = [df[(df.R == R) & (df.method == m)][k].mean() for R in Rs]
            bars = a.bar(x + (i - (len(methods) - 1) / 2) * w, vals, w, label=m)
            a.bar_label(bars, fmt="%.1f" if k == "psnr" else "%.3f", fontsize=7)
        a.set_xticks(x); a.set_xticklabels([f"R = {R}" for R in Rs])
        a.set_title(f"{NAMES[k]} ({'higher' if HIGHER[k] else 'lower'} is better)")
        if k == "psnr":
            a.set_ylim(df[df.method.isin(methods)].groupby(["R", "method"]).psnr.mean().min() - 3, None)
    ax[0].legend(fontsize=8)
    fig.suptitle(title)
    plt.tight_layout(); save(fig, fname)

def box_plots(df, methods, fname, title):
    Rs = sorted(df.R.unique())
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
    for a, k in zip(ax, ["psnr", "ssim_fg", "nmse"]):
        data = [df[(df.R == R) & (df.method == m)][k].values for R in Rs for m in methods]
        a.boxplot(data)
        a.set_xticks(range(1, len(data) + 1))
        a.set_xticklabels([f"{m}\nR={R}" for R in Rs for m in methods], fontsize=7)
        a.set_title(f"{NAMES[k]} per test case ({'higher' if HIGHER[k] else 'lower'} is better)")
    fig.suptitle(title)
    plt.tight_layout(); save(fig, fname)

def subject_chart(df, methods, fname, title):
    Rs, subs = sorted(df.R.unique()), sorted(df.subject.unique())
    x, w = np.arange(len(subs)), 0.8 / len(methods)
    fig, ax = plt.subplots(1, len(Rs), figsize=(7.5 * len(Rs), 4.4), squeeze=False)
    for a, R in zip(ax[0], Rs):
        for i, m in enumerate(methods):
            vals = [df[(df.R == R) & (df.method == m) & (df.subject == s)].psnr.mean() for s in subs]
            a.bar(x + (i - (len(methods) - 1) / 2) * w, vals, w, label=m)
        a.set_xticks(x); a.set_xticklabels([f"subject {s}" for s in subs])
        a.set_ylim(df[(df.R == R) & df.method.isin(methods)].groupby(["subject", "method"]).psnr.mean().min() - 2, None)
        a.set_title(f"mean PSNR (dB) per test subject, R = {R}")
    ax[0, 0].legend(fontsize=8)
    fig.suptitle(title)
    plt.tight_layout(); save(fig, fname)

METHODS = ["zero-filled", "per-coil", "multi-coil"]
main = ev[(ev.pattern == "vd") & ev.R.isin(R_MAIN)]
table2 = means_table(main, METHODS, ["zero-filled"])
table2.to_csv(os.path.join(RES_DIR, "table2_main_results.csv"), index=False)
print("Mean scores on the test subjects (variable-density masks)")
display(table2)
bar_chart(main, METHODS, "fig07a_methods_bars.png", "Zero-filled vs per-coil FISTA vs multi-coil FISTA (test subjects)")
box_plots(main, METHODS, "fig07b_methods_boxplots.png", "Spread over test slices and masks")
subject_chart(main, METHODS, "fig07c_methods_per_subject.png", "The same comparison for each test subject")
for R in R_MAIN:
    m = main[main.R == R].groupby("method").psnr.mean()
    best = m.idxmax()
    d = paired(main[main.R == R], best, "zero-filled")
    print(f"R = {R}: best method {best} ({m[best]:.2f} dB), {d.psnr.mean():+.2f} dB over zero-filled, better on {(d.psnr > 0).sum()} of {len(d)} cases")

**Reading the charts:** bars are means over all test cases (slices × masks); box plots show the spread; the per-subject chart shows whether the ranking holds for every test person. *Better than* counts the cases (same slice, same mask) where a method beats zero-filled.

In [ ]:
curve = ev[ev.pattern == "vd"]
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
for j, (metric, label) in enumerate([("psnr", "PSNR (dB)"), ("ssim_fg", "SSIM (knee)"), ("nmse", "NMSE (lower is better)")]):
    for m, style in [("zero-filled", "o--"), ("multi-coil", "o-"), ("per-coil", "s")]:
        g = curve[curve.method == m].groupby("R")[metric]
        ax[j].errorbar(g.mean().index, g.mean().values, yerr=g.std().values, fmt=style, capsize=3, label=m)
    ax[j].set_xlabel("acceleration factor R"); ax[j].set_ylabel(label); ax[j].set_xticks(R_CURVE); ax[j].legend()
ax[0].set_title("Quality vs acceleration (mean ± sd, test subjects)")
plt.tight_layout()
save(fig, "fig07_quality_vs_acceleration.png")

curve_table = curve.groupby(["R", "method"])["psnr"].mean().unstack("method")
curve_table.to_csv(os.path.join(RES_DIR, "table3_quality_vs_acceleration.csv"))
print("Mean PSNR (dB) by acceleration")
display(curve_table.round(2))

**Reading Figure 7:** each line shows how image quality falls as fewer lines are kept. The vertical gap between zero-filled and a reconstruction method is what reconstruction recovers at that speed. Per-coil FISTA was only run at `R_MAIN`; multi-coil FISTA has its own tuned setting at every R.

In [ ]:
pat = ev[ev.R.isin(R_MAIN) & ev.method.isin(["zero-filled", "multi-coil"])]
rows = []
for R in R_MAIN:
    for p in PATTERNS:
        d = pat[(pat.R == R) & (pat.pattern == p)]
        diff = paired(d, "multi-coil")
        rows.append({"R": R, "pattern": p, "cases": len(diff),
                     "zero-filled PSNR": d[d.method == "zero-filled"].psnr.mean(),
                     "multi-coil PSNR": d[d.method == "multi-coil"].psnr.mean(),
                     "Δ PSNR (dB)": diff.psnr.mean(), "Δ SSIM (knee)": diff.ssim_fg.mean(),
                     "better PSNR": f"{(diff.psnr > 0).sum()}/{len(diff)}"})
table4 = pd.DataFrame(rows)
table4.to_csv(os.path.join(RES_DIR, "table4_sampling_pattern.csv"), index=False)
print("Sampling pattern x reconstruction (mean PSNR, dB)")
display(table4.round(3))

fig, ax = plt.subplots(1, len(R_MAIN), figsize=(7 * len(R_MAIN), 4.3), squeeze=False)
for a, R in zip(ax[0], R_MAIN):
    t = table4[table4.R == R]
    x = np.arange(len(t))
    b1 = a.bar(x - 0.2, t["zero-filled PSNR"], 0.4, label="zero-filled")
    b2 = a.bar(x + 0.2, t["multi-coil PSNR"], 0.4, label="multi-coil FISTA")
    a.bar_label(b1, fmt="%.1f", fontsize=8); a.bar_label(b2, fmt="%.1f", fontsize=8)
    a.set_xticks(x); a.set_xticklabels(t["pattern"].replace({"vd": "variable density"}))
    a.set_ylim(min(t["zero-filled PSNR"].min(), t["multi-coil PSNR"].min()) - 3, None)
    a.set_title(f"mean PSNR (dB) by sampling pattern, R = {R}")
ax[0, 0].legend()
plt.tight_layout()
save(fig, "fig07d_sampling_patterns.png")

**Reading Table 4:** the PSF argument of Phase 4 predicts that reconstruction helps incoherent (random, variable-density) sampling more than equispaced sampling. With a large ACS block the differences may be small, since most aliasing is then blur rather than ghosts. Equispaced masks are deterministic, so they are evaluated with one seed only.

In [ ]:
diag = ev[(ev.pattern == "vd") & ev.R.isin(R_MAIN) & (ev.seed == SEEDS[0])]
table5 = (diag.groupby(["R", "method"])[["psnr", "ssim", "ssim_fg"]].mean()
          .rename(columns={"ssim": "SSIM whole image", "ssim_fg": "SSIM inside knee"}))
table5["SSIM gap (whole − knee)"] = table5["SSIM whole image"] - table5["SSIM inside knee"]
table5.to_csv(os.path.join(RES_DIR, "table5_ssim_and_maps.csv"))
print("Table 5: whole-image vs knee-only SSIM, and ESPIRiT vs low-res maps (first seed)")
display(table5.round(4))

**Reading Table 5:** two possible reasons for a reconstruction to score lower on SSIM than expected are checked here. (1) **Background:** the ground truth background is pure scanner noise; a method that suppresses it changes local statistics there, which lowers whole-image SSIM without affecting the anatomy. If the drop disappears inside the knee, that explains it. (2) **Coil maps:** comparing multi-coil with ESPIRiT maps against low-res maps shows how much the quality of the coil model matters.

In [ ]:
subj_v, pos_v, R_v = TEST[0], n_saved(TEST[0]) // 2, max(R_MAIN)
ks_v, ref_v = get_slice(subj_v, pos_v)
fg_v = foreground_mask(ref_v)
m_v = make_mask(ks_v.shape[1], R_v, "vd", SEEDS[0])
y_v = undersample(ks_v, m_v)
imgs = {"zero-filled": recon_zf(y_v),
        "per-coil FISTA": recon_pc(y_v, m_v, *PC_SET[R_v])[0],
        "multi-coil FISTA": recon_mc(y_v, m_v, espirit_maps(ks_v, **ESPIRIT)[0], *MC_SET[R_v])[0]}
fig, ax = plt.subplots(2, 4, figsize=(17, 9))
show(ax[0, 0], ref_v, f"ground truth (subject {subj_v}, slice {subject_data(subj_v)['slices'][pos_v]})")
ax[1, 0].axis("off")
for j, (name, img) in enumerate(imgs.items(), start=1):
    s = metrics(img, ref_v, fg_v)
    show(ax[0, j], img, f"{name}, R = {R_v}\nPSNR {s['psnr']:.1f} dB  SSIM(knee) {s['ssim_fg']:.3f}")
    show(ax[1, j], np.abs(img - ref_v), "absolute error", vmax=0.3, cmap="inferno")
plt.tight_layout()
save(fig, f"fig08_visual_comparison_R{R_v}.png")

**Reading Figure 8:** a held-out slice at the highest acceleration. The error maps show *where* each method fails: errors along edges mean lost detail, errors spread over smooth regions mean residual aliasing or noise. Higher PSNR only means real recovery if the error drops inside the anatomy, not just in the background.

## Phase 8 · Coil-count experiment
If coil redundancy is what makes high acceleration recoverable, quality should rise with the number of coils. At R = `COIL_R`, the multi-coil reconstruction is repeated using only 1, 2, 4 or all coils (random subsets, `COIL_SUBSETS` per size). For a fair comparison, each subset is compared with **its own** fully sampled ground truth (RSS of the same coils), and its coil maps are estimated from those coils only. Every second saved slice of each test subject is used.

In [ ]:
def run_coils():
    rows = []
    rng = np.random.default_rng(7)
    for subj in TEST:
        nc_s = subject_data(subj)["kspace"].shape[1]
        for pos in tqdm(range(0, n_saved(subj), 2), desc=f"subject {subj}"):
            for count in [c for c in COIL_COUNTS if c <= nc_s]:
                subsets = ([tuple(range(nc_s))] if count == nc_s else
                           [tuple(sorted(int(i) for i in rng.choice(nc_s, count, replace=False))) for _ in range(COIL_SUBSETS)])
                for sub in subsets:
                    ks_c, ref_c = get_slice(subj, pos, coils=sub)
                    fg_c = foreground_mask(ref_c)
                    mask = make_mask(ks_c.shape[1], COIL_R, "vd", SEEDS[0])
                    y = undersample(ks_c, mask)
                    S = espirit_maps(ks_c, **ESPIRIT)[0]
                    for method, img in (("zero-filled", recon_zf(y)), ("multi-coil", recon_mc(y, mask, S, *MC_SET[COIL_R])[0])):
                        rows.append({"subject": subj, "pos": pos, "coils": count, "subset": "-".join(map(str, sub)),
                                     "method": method, **metrics(img, ref_c, fg_c)})
    return rows

t0 = time.time()
coil_df = pd.DataFrame(cached("coil_count.json", run_coils))
print(f"coil experiment: {len(coil_df)} rows ({time.time() - t0:.0f} s)")

piv = coil_df.pivot_table(index=["subject", "pos", "coils", "subset"], columns="method", values=["psnr", "ssim_fg"])
gain = pd.DataFrame({"coils": piv.index.get_level_values("coils"),
                     "Δ PSNR": (piv["psnr"]["multi-coil"] - piv["psnr"]["zero-filled"]).values,
                     "Δ SSIM (knee)": (piv["ssim_fg"]["multi-coil"] - piv["ssim_fg"]["zero-filled"]).values})
table6 = coil_df.groupby(["coils", "method"])[["psnr", "ssim_fg"]].mean().unstack("method")
table6[("gain", "Δ PSNR (dB)")] = gain.groupby("coils")["Δ PSNR"].mean()
table6[("gain", "Δ SSIM (knee)")] = gain.groupby("coils")["Δ SSIM (knee)"].mean()
table6[("gain", "better PSNR")] = gain.groupby("coils")["Δ PSNR"].apply(lambda d: f"{(d > 0).sum()}/{len(d)}")
table6.to_csv(os.path.join(RES_DIR, "table6_coil_count.csv"))
print(f"Number of coils at R = {COIL_R}")
display(table6.round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
for m, style in [("zero-filled", "o--"), ("multi-coil", "o-")]:
    g = coil_df[coil_df.method == m].groupby("coils")["psnr"]
    ax[0].errorbar(g.mean().index, g.mean().values, yerr=g.std().values, fmt=style, capsize=3, label=m)
g = gain.groupby("coils")["Δ PSNR"]
ax[1].errorbar(g.mean().index, g.mean().values, yerr=g.std().values, fmt="o-", capsize=3, color="C2")
ax[1].axhline(0, color="gray", lw=0.8)
for a, title in zip(ax, [f"PSNR vs number of coils (R = {COIL_R})", "Gain of multi-coil FISTA over zero-filled"]):
    a.set_xscale("log", base=2); a.set_xticks(COIL_COUNTS); a.set_xticklabels(COIL_COUNTS)
    a.set_xlabel("coils used"); a.set_title(title)
ax[0].set_ylabel("PSNR (dB)"); ax[1].set_ylabel("Δ PSNR (dB)"); ax[0].legend()
plt.tight_layout()
save(fig, "fig09_coil_count.png")

**Reading Table 6 / Figure 9:** the right panel isolates what the reconstruction adds for each number of coils. A gain that grows with coil count supports the idea that the coils act as extra measurements; a flat gain would mean sparsity alone is doing the work. With a single coil the method reduces to plain single-coil compressed sensing.

## Phase 9 · Motion: simulation and blind correction
**Fourier shift theorem:** translating an image by (dy, dx) pixels multiplies its k-space by a linear phase, exp(−2πi(f_y·dy + f_x·dx)). If the patient moves during the scan, the lines acquired after the movement carry that phase and the earlier lines do not. The two groups describe slightly different images, and the inverse FFT turns the inconsistency into ghosting and doubled edges.

**Simulation model (assumptions):**
* rigid in-plane translation at one moment (the *onset*), applied to fully sampled data;
* ky lines are acquired in increasing ky order, so "onset at line *o*" means lines *o* and later are shifted (the real acquisition order of this scan is not modelled);
* the same phase is applied to every coil (the coils are treated as moving with the knee).

**Blind correction (autofocus):** without knowing the motion, try candidate onsets and shifts, undo the corresponding phase on the later lines, and keep the candidate that gives the **sharpest** image. Sharpness is measured by **image entropy**, which is low for crisp images and high for blurred or ghosted ones. To make the search fast, the coils are first combined into one *virtual coil* (a fixed weighted sum of coils, which does not change the phase relationship). The search runs coarse (integer shifts via exact circular shifts of the two image parts), then fine (onset ±16 lines), then continuous (Nelder–Mead on the shift). The continuous step is only accepted if it stays inside the search range and gives a sharper image than the integer estimate it started from.

In [ ]:
def motion_ramp(ny, nx, dy, dx):
    fy = (np.arange(ny, dtype=np.float32) - ny // 2) / ny
    fx = (np.arange(nx, dtype=np.float32) - nx // 2) / nx
    return np.exp(-2j * np.pi * (fy[:, None] * dy + fx[None, :] * dx)).astype(np.complex64)

def corrupt(ks, onset, dy, dx):
    out = ks.copy()
    out[..., onset:, :] *= motion_ramp(ks.shape[-2], ks.shape[-1], dy, dx)[onset:]
    return out

def correct(ks, onset, dy, dx):
    out = ks.copy()
    out[..., onset:, :] *= np.conj(motion_ramp(ks.shape[-2], ks.shape[-1], dy, dx)[onset:])
    return out

def motion_positions(subj):
    n = n_saved(subj)
    return sorted(set(np.round(np.linspace(0, n - 1, MOTION_SLICES_PER_SUBJECT + 2)[1:-1]).astype(int).tolist()))

kc = corrupt(ks, ny // 3, 4.0, 2.0)
print(f"oracle check (undo the known motion): max error {np.abs(rss(ifft2c(correct(kc, ny // 3, 4.0, 2.0))) - ref).max():.1e}")

### 9a · How severe is motion?
Shifts of 1, 2, 4 and 8 pixels in y (half that in x), with the onset moved from 10 % to 90 % of the scan, on `MOTION_SLICES_PER_SUBJECT` slices per test subject.

In [ ]:
ONSETS = [round(f, 1) for f in np.arange(0.1, 0.91, 0.1)]
SHIFTS = [1, 2, 4, 8]

def run_severity():
    rows = []
    for subj in TEST:
        for pos in motion_positions(subj):
            ks_m, ref_m = get_slice(subj, pos)
            fg_m = foreground_mask(ref_m)
            for f in ONSETS:
                for d in SHIFTS:
                    img = recon_zf(corrupt(ks_m, int(f * ks_m.shape[1]), d, d / 2))
                    rows.append({"subject": subj, "pos": pos, "onset": f, "shift": d, **metrics(img, ref_m, fg_m)})
    return rows

sev = pd.DataFrame(cached("motion_severity.json", run_severity))
table7 = sev.groupby(["onset", "shift"])["psnr"].mean().unstack("shift")
table7.columns = [f"{c} px" for c in table7.columns]
table7.to_csv(os.path.join(RES_DIR, "table7_motion_severity_psnr.csv"))
print("Table 7: PSNR (dB) of motion-corrupted images, by onset (fraction of scan) and shift")
display(table7)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
for d in SHIFTS:
    g = sev[sev["shift"] == d].groupby("onset")
    ax[0].plot(np.array(ONSETS) * 100, g["psnr"].mean().values, "o-", label=f"{d} px (y), {d / 2:g} px (x)")
    ax[1].plot(np.array(ONSETS) * 100, g["ssim_fg"].mean().values, "o-", label=f"{d} px")
ax[0].set_ylabel("PSNR (dB)"); ax[1].set_ylabel("SSIM inside the knee")
for a in ax:
    a.set_xlabel("motion onset (% of the scan)"); a.axvline(50, color="gray", ls=":", lw=1); a.legend(fontsize=8)
ax[0].set_title("Motion severity vs onset and shift (dotted: k-space centre)")
plt.tight_layout()
save(fig, "fig10_motion_severity.png")

subj_m, pos_m = TEST[0], motion_positions(TEST[0])[0]
ks_m, ref_m = get_slice(subj_m, pos_m)
fig, ax = plt.subplots(2, 4, figsize=(17, 9))
show(ax[0, 0], ref_m, "ground truth"); ax[1, 0].axis("off")
for j, f in enumerate([0.25, 0.5, 0.75], start=1):
    img = recon_zf(corrupt(ks_m, int(f * ks_m.shape[1]), 4, 2))
    show(ax[0, j], img, f"4 px motion at {f:.0%} of the scan  PSNR {metrics(img, ref_m, foreground_mask(ref_m))['psnr']:.1f} dB")
    show(ax[1, j], np.abs(img - ref_m), "absolute error", vmax=0.3, cmap="inferno")
plt.tight_layout()
save(fig, "fig11_motion_examples.png")

**Reading Table 7 / Figures 10–11:** severity depends mostly on *when* the motion happens. Motion before the centre of k-space (left of the dotted line) corrupts the energy-heavy central lines; late motion only touches outer lines that carry little energy. For early onset, most lines are shifted, so the image is largely a displaced copy of the knee; part of the PSNR loss is then misregistration against the reference rather than blur. Because a phase is only defined modulo one full turn, a larger pixel shift does not always mean a larger phase error on a given line, so the curves for different shifts can cross at late onsets.

### 9b · Blind motion correction
For each motion slice, `MOTION_CASES_PER_SLICE` random motions are simulated (onset between 20 % and 80 % of the scan, shifts up to ±`MOTION_MAX_SHIFT` px in each direction, at least 1.5 px in total). The autofocus searches onsets over the whole scan and shifts up to ±`MOTION_SEARCH` px, without being told the true values.

In [ ]:
def virtual_coil(ks):
    flat = ks.reshape(ks.shape[0], -1)
    _, v = np.linalg.eigh(flat @ flat.conj().T)
    return np.tensordot(v[:, -1].conj(), ks, axes=(0, 0))

def entropy(img):
    a = np.abs(img).astype(np.float64)
    b = a / np.sqrt((a ** 2).sum(axis=(-2, -1), keepdims=True))
    return -(b * np.log(b + 1e-12)).sum(axis=(-2, -1))

def autofocus(ks, search=MOTION_SEARCH, onset_step=16):
    v = virtual_coil(ks)
    ny_v = v.shape[0]

    def best_integer(o, ys, xs):
        kb = np.zeros_like(v)
        kb[o:] = v[o:]
        ia, ib = ifft2c(v - kb), ifft2c(kb)
        return min((float(entropy(ia + np.roll(ib, (-dy, -dx), axis=(0, 1)))), o, dy, dx) for dy in ys for dx in xs)

    def cost(p, o):
        return float(entropy(ifft2c(correct(v, o, float(p[0]), float(p[1])))))

    full = range(-search, search + 1)
    _, o0, dy0, dx0 = min(best_integer(o, full, full) for o in range(ny_v // 8, ny_v - ny_v // 8 + 1, onset_step))
    near_y = range(max(dy0 - 2, -search), min(dy0 + 2, search) + 1)
    near_x = range(max(dx0 - 2, -search), min(dx0 + 2, search) + 1)
    _, o1, dy1, dx1 = min(best_integer(o, near_y, near_x)
                          for o in range(max(o0 - onset_step, 1), min(o0 + onset_step, ny_v - 1) + 1))
    res = optimize.minimize(cost, [dy1, dx1], args=(o1,), method="Nelder-Mead",
                            options={"xatol": 0.01, "fatol": 1e-10, "maxiter": 200,
                                     "initial_simplex": [[dy1, dx1], [dy1 + 0.5, dx1], [dy1, dx1 + 0.5]]})
    candidates = [(cost(res.x, o1), float(res.x[0]), float(res.x[1])), (cost((dy1, dx1), o1), float(dy1), float(dx1))]
    _, dy2, dx2 = min(c for c in candidates if max(abs(c[1]), abs(c[2])) <= search + 0.5)
    o2 = min(range(max(o1 - 2, 1), min(o1 + 2, ny_v - 1) + 1), key=lambda o: cost((dy2, dx2), o))
    return o2, dy2, dx2

def run_blind():
    rows = []
    rng = np.random.default_rng(2024)
    cases = [(s, p) for s in TEST for p in motion_positions(s) for _ in range(MOTION_CASES_PER_SLICE)]
    for subj, pos in tqdm(cases, desc="motion cases"):
        ks_m, ref_m = get_slice(subj, pos)
        fg_m = foreground_mask(ref_m)
        n = ks_m.shape[1]
        onset = int(rng.integers(int(0.2 * n), int(0.8 * n) + 1))
        while True:
            dy, dx = (float(v) for v in rng.uniform(-MOTION_MAX_SHIFT, MOTION_MAX_SHIFT, 2))
            if math.hypot(dy, dx) >= 1.5:
                break
        kc = corrupt(ks_m, onset, dy, dx)
        t0 = time.time()
        o_e, dy_e, dx_e = autofocus(kc)
        seconds = time.time() - t0
        before = metrics(recon_zf(kc), ref_m, fg_m)
        after = metrics(recon_zf(correct(kc, o_e, dy_e, dx_e)), ref_m, fg_m)
        rows.append({"subject": subj, "pos": pos, "onset": onset, "onset_frac": onset / n, "dy": dy, "dx": dx,
                     "onset_est": o_e, "dy_est": dy_e, "dx_est": dx_e,
                     "shift_error_px": math.hypot(dy_e - dy, dx_e - dx), "onset_error_lines": abs(o_e - onset),
                     "psnr_before": before["psnr"], "psnr_after": after["psnr"],
                     "ssim_fg_before": before["ssim_fg"], "ssim_fg_after": after["ssim_fg"], "seconds": seconds})
    return rows

blind = pd.DataFrame(cached("motion_blind.json", run_blind))
blind.to_csv(os.path.join(RES_DIR, "table8_motion_blind_cases.csv"), index=False)
summary8 = pd.Series({
    "cases": len(blind),
    "median shift error (px)": blind.shift_error_px.median(),
    "cases with shift error < 0.5 px": f"{(blind.shift_error_px < 0.5).sum()}/{len(blind)}",
    "median onset error (lines)": blind.onset_error_lines.median(),
    "mean PSNR before (dB)": blind.psnr_before.mean(),
    "mean PSNR after (dB)": blind.psnr_after.mean(),
    "cases improved (PSNR)": f"{(blind.psnr_after > blind.psnr_before).sum()}/{len(blind)}",
    "mean SSIM (knee) before": blind.ssim_fg_before.mean(),
    "mean SSIM (knee) after": blind.ssim_fg_after.mean(),
    "mean search time (s)": blind.seconds.mean()}, name="value")
summary8.to_frame().to_csv(os.path.join(RES_DIR, "table8_motion_blind_summary.csv"))
print("Table 8: blind motion correction")
display(summary8.to_frame())
display(blind[["subject", "onset", "onset_est", "dy", "dy_est", "dx", "dx_est", "shift_error_px",
               "psnr_before", "psnr_after"]].sort_values("onset").round(2))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
lim = MOTION_SEARCH
for a, true, est, name in [(ax[0], blind.dy, blind.dy_est, "dy"), (ax[0], blind.dx, blind.dx_est, "dx")]:
    a.plot(true, est, "o", ms=5, alpha=0.8, label=name)
ax[0].plot([-lim, lim], [-lim, lim], "k--", lw=0.8)
ax[0].set_xlabel("true shift (px)"); ax[0].set_ylabel("estimated shift (px)"); ax[0].legend()
ax[0].set_title("Estimated vs true shift")
ax[1].plot(blind.onset_frac * 100, blind.shift_error_px, "o")
ax[1].set_xlabel("true onset (% of scan)"); ax[1].set_ylabel("shift error (px)"); ax[1].set_yscale("log")
ax[1].set_title("Shift error vs onset")
ax[2].plot(blind.psnr_before, blind.psnr_after, "o")
lo_, hi_ = blind[["psnr_before", "psnr_after"]].min().min() - 1, blind[["psnr_before", "psnr_after"]].max().max() + 1
ax[2].plot([lo_, hi_], [lo_, hi_], "k--", lw=0.8)
ax[2].set_xlabel("PSNR before (dB)"); ax[2].set_ylabel("PSNR after blind correction (dB)")
ax[2].set_title("Image quality before vs after")
plt.tight_layout()
save(fig, "fig12_motion_blind_accuracy.png")

case = blind.loc[blind.psnr_before.idxmin()]
ks_c, ref_c = get_slice(int(case.subject), int(case.pos))
kc = corrupt(ks_c, int(case.onset), case.dy, case.dx)
fixed = recon_zf(correct(kc, int(case.onset_est), case.dy_est, case.dx_est))
v = virtual_coil(kc)
kb = np.zeros_like(v); kb[int(case.onset_est):] = v[int(case.onset_est):]
ia, ib = ifft2c(v - kb), ifft2c(kb)
grid = np.arange(-MOTION_SEARCH, MOTION_SEARCH + 1)
landscape = np.array([[entropy(ia + np.roll(ib, (-a, -b), axis=(0, 1))) for b in grid] for a in grid])

fig, ax = plt.subplots(2, 3, figsize=(15, 9.5))
show(ax[0, 0], recon_zf(kc), f"corrupted  PSNR {case.psnr_before:.1f} dB")
show(ax[0, 1], fixed, f"blind correction  PSNR {case.psnr_after:.1f} dB")
show(ax[0, 2], ref_c, "ground truth")
show(ax[1, 0], np.abs(recon_zf(kc) - ref_c), "error before", vmax=0.3, cmap="inferno")
show(ax[1, 1], np.abs(fixed - ref_c), "error after", vmax=0.3, cmap="inferno")
im = ax[1, 2].imshow(landscape, extent=[grid[0] - 0.5, grid[-1] + 0.5, grid[-1] + 0.5, grid[0] - 0.5], cmap="viridis")
ax[1, 2].plot(case.dx, case.dy, "r+", ms=14, mew=2, label="true")
ax[1, 2].plot(case.dx_est, case.dy_est, "wx", ms=10, mew=2, label="estimated")
ax[1, 2].set_xlabel("dx (px)"); ax[1, 2].set_ylabel("dy (px)"); ax[1, 2].legend(fontsize=8)
ax[1, 2].set_title(f"entropy over integer shifts (onset line {int(case.onset_est)})")
plt.colorbar(im, ax=ax[1, 2], fraction=0.046)
fig.suptitle(f"Worst corrupted case: true onset {int(case.onset)}, shift ({case.dy:.2f}, {case.dx:.2f}) px; "
             f"estimated onset {int(case.onset_est)}, shift ({case.dy_est:.2f}, {case.dx_est:.2f}) px")
plt.tight_layout()
save(fig, "fig13_motion_blind_example.png")

**Reading Table 8 / Figures 12–13:** points on the diagonal mean the shift was recovered. Because the simulated motion is a pure phase, an accurate estimate removes it almost exactly, so *PSNR after* can be very high and mainly reflects the small remaining estimation error; the shift and onset errors are the meaningful measures. The middle panel shows whether accuracy depends on when the motion happened: late motion touches only low-energy outer lines, so it changes the image (and the entropy) little, which makes it harder to estimate, but also less harmful when missed. The entropy landscape of the worst case should show a clear minimum near the true shift; a flat or multi-valley landscape signals where this criterion struggles.

## Phase 10 · Summary
The cell below collects the main numbers from all phases and lists every saved figure and table.

In [ ]:
print("Q1  Where the information lives")
r0 = table1.iloc[0]
print(f"    A central disk of radius {int(r0['radius'])} ({r0['% of k-space points']:.2f}% of k-space points) "
      f"holds {r0['% of signal energy']:.1f}% of the signal energy.")

print("\nQ2/Q3  Undersampling and reconstruction (held-out test subjects, variable-density masks)")
for R in R_MAIN:
    dR = main[main.R == R]
    zf_p = dR[dR.method == "zero-filled"].psnr.mean()
    line = f"    R = {R}: zero-filled {zf_p:.2f} dB"
    for m in ("per-coil", "multi-coil"):
        d = paired(dR, m)
        line += f" | {m} {zf_p + d.psnr.mean():.2f} dB (Δ {d.psnr.mean():+.2f} dB, better on {(d.psnr > 0).sum()}/{len(d)})"
    print(line)
cm = curve[curve.method == "multi-coil"].groupby("R").psnr.mean()
cz = curve[curve.method == "zero-filled"].groupby("R").psnr.mean()
print("    Multi-coil gain over zero-filled by R: " + ", ".join(f"R={R}: {cm[R] - cz[R]:+.2f} dB" for R in R_CURVE))
g6 = gain.groupby("coils")["Δ PSNR"].mean()
print(f"    Coil count at R = {COIL_R}: multi-coil gain " + ", ".join(f"{c} coil(s): {g6[c]:+.2f} dB" for c in g6.index))

print("\nQ4  Motion")
print(f"    4 px motion: PSNR {table7.loc[ONSETS[0], '4 px']:.1f} dB at onset {ONSETS[0]:.0%} vs "
      f"{table7.loc[ONSETS[-1], '4 px']:.1f} dB at onset {ONSETS[-1]:.0%}")
print(f"    Blind correction: median shift error {blind.shift_error_px.median():.2f} px, "
      f"{(blind.shift_error_px < 0.5).sum()}/{len(blind)} cases within 0.5 px, "
      f"PSNR {blind.psnr_before.mean():.1f} -> {blind.psnr_after.mean():.1f} dB")

print(f"\nNotebook 2 input: {len(os.listdir(UNET_DIR))} U-Net subject files in unet_data/, "
      f"{len(TEST)} test subjects in data/, settings in results/project_config.json and results/frozen_settings.csv")
print("\nFigures:")
for f in sorted(os.listdir(FIG_DIR)):
    print("   ", f)
print("Results:")
for f in sorted(os.listdir(RES_DIR)):
    print("   ", f)

### Scope and assumptions
* **Retrospective undersampling** of fully sampled data from healthy volunteers on one scanner; a truly faster acquisition can also change contrast and noise.
* Algorithm settings were chosen on one subject and frozen; they are reasonable, not optimal for every subject or acceleration.
* The **motion model** is a single rigid in-plane shift with ky-ordered acquisition and the same phase in every coil; real motion also includes rotation, through-plane motion and a scanner-specific line order.
* Metrics are computed against the RSS of real data, which contains scanner noise; no method can match that noise, which caps the achievable PSNR.

### References
* Lustig M, Donoho D, Pauly JM. *Sparse MRI: The application of compressed sensing for rapid MR imaging.* Magn Reson Med, 2007.
* Beck A, Teboulle M. *A fast iterative shrinkage-thresholding algorithm for linear inverse problems.* SIAM J Imaging Sci, 2009.
* Pruessmann KP, Weiger M, Scheidegger MB, Boesiger P. *SENSE: Sensitivity encoding for fast MRI.* Magn Reson Med, 1999.
* Uecker M et al. *ESPIRiT — an eigenvalue approach to autocalibrating parallel MRI: where SENSE meets GRAPPA.* Magn Reson Med, 2014.
* Atkinson D et al. *Automatic correction of motion artifacts in magnetic resonance images using an entropy focus criterion.* IEEE Trans Med Imaging, 1997.
* Wang Z, Bovik AC, Sheikh HR, Simoncelli EP. *Image quality assessment: from error visibility to structural similarity.* IEEE Trans Image Process, 2004.
* Zbontar J et al. *fastMRI: An open dataset and benchmarks for accelerated MRI.* arXiv:1811.08839, 2018.
* Data: *Stanford Fully Sampled 3D FSE Knees*, mridata.org (Ong F et al., *mridata.org: An open archive for sharing MRI raw data*, ISMRM 2018; Epperson K et al., SMRT 2013).